In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/MajorProjectModel4_price_prediction/Price_Agriculture_commodities_Week.csv")

df.shape

(23093, 10)

In [ ]:
df.head()


,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min Price,Max Price,Modal Price
0,Gujarat,Amreli,Damnagar,Bhindi(Ladies Finger),Bhindi,FAQ,27-07-2023,4100.0,4500.0,4350.0
1,Gujarat,Amreli,Damnagar,Brinjal,Other,FAQ,27-07-2023,2200.0,3000.0,2450.0
2,Gujarat,Amreli,Damnagar,Cabbage,Cabbage,FAQ,27-07-2023,2350.0,3000.0,2700.0
3,Gujarat,Amreli,Damnagar,Cauliflower,Cauliflower,FAQ,27-07-2023,7000.0,7500.0,7250.0
4,Gujarat,Amreli,Damnagar,Coriander(Leaves),Coriander,FAQ,27-07-2023,8400.0,9000.0,8850.0


In [ ]:
!pip install catboost joblib pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.2 MB/s eta 0:00:00


In [ ]:
# ✅ PRICE PREDICTION PIPELINE — FINAL VERSION
# Install dependencies if needed:
# !pip install catboost joblib pandas scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor, Pool
import joblib

# ================= 1️⃣ Load Data =================
file_path = "/content/drive/MyDrive/MajorProjectModel4_price_prediction/Price_Agriculture_commodities_Week.csv"
df = pd.read_csv(file_path)
print("Data Loaded ✅ Rows:", len(df))

# ================= 2️⃣ Clean + Feature Engineering =================
# Convert prices to numeric
df["Min Price"] = pd.to_numeric(df["Min Price"], errors="coerce")
df["Max Price"] = pd.to_numeric(df["Max Price"], errors="coerce")
df["Modal Price"] = pd.to_numeric(df["Modal Price"], errors="coerce")

# Remove rows without target price
df = df.dropna(subset=["Modal Price"]).reset_index(drop=True)

# Convert Arrival_Date to datetime
df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"], dayfirst=True, errors="coerce")
df["Arrival_Date"] = df["Arrival_Date"].fillna(pd.to_datetime("1970-01-01"))

# Extract useful date-based features
df["arrival_day"] = df["Arrival_Date"].dt.day
df["arrival_month"] = df["Arrival_Date"].dt.month
df["arrival_year"] = df["Arrival_Date"].dt.year
df["arrival_dayofweek"] = df["Arrival_Date"].dt.dayofweek

# Price spread feature
df["price_spread"] = df["Max Price"] - df["Min Price"]

# ================= 3️⃣ Features & Target =================
categorical_cols = ["State", "District", "Market", "Commodity", "Variety", "Grade"]
numeric_cols = ["Min Price", "Max Price", "price_spread",
                "arrival_day", "arrival_month", "arrival_year", "arrival_dayofweek"]

X = df[categorical_cols + numeric_cols]
y = df["Modal Price"]

# ================= 4️⃣ Train-Test Split =================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
test_pool = Pool(X_test, y_test, cat_features=categorical_cols)

# ================= 5️⃣ Train CatBoost Model =================
model = CatBoostRegressor(
    iterations=600,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

model.fit(train_pool, eval_set=test_pool)

# ================= 6️⃣ Evaluation =================
preds = model.predict(test_pool)
mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds)

print("\n✅ Model Evaluation:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

# Show a few predictions
preview = X_test.copy().reset_index(drop=True)
preview["Actual Price"] = y_test.reset_index(drop=True)
preview["Predicted Price"] = np.round(preds, 2)
print("\nSample Predictions:")
print(preview.head())

# ================= 7️⃣ Save Model =================
model_path = "/content/drive/MyDrive/MajorProjectModel4_price_prediction/price_model.cbm"
model.save_model(model_path)
joblib.dump(categorical_cols + numeric_cols, "features_list.joblib")

print("\n✅ Model Saved:", model_path)
print("✅ Feature List Saved: features_list.joblib")

# ================= 8️⃣ Example Prediction (Load + Predict) =================
print("""
To Predict Later:

from catboost import CatBoostRegressor
import pandas as pd
import joblib

model = CatBoostRegressor()
model.load_model("price_model.cbm")
features = joblib.load("features_list.joblib")

sample = pd.DataFrame([{
    "State": "Gujarat",
    "District": "Amreli",
    "Market": "Damnagar",
    "Commodity": "Tomato",
    "Variety": "Local",
    "Grade": "FAQ",
    "Min Price": 9800,
    "Max Price": 10000,
    "price_spread": 200,
    "arrival_day": 27,
    "arrival_month": 7,
    "arrival_year": 2023,
    "arrival_dayofweek": 3
}])[features]

pred = model.predict(sample)
print(pred)
""")


Data Loaded ✅ Rows: 23093
0:	learn: 5676.9158605	test: 5329.4470696	best: 5329.4470696 (0)	total: 49.3ms	remaining: 29.6s
100:	learn: 817.2086754	test: 1262.5725449	best: 1262.5725449 (100)	total: 4.2s	remaining: 20.7s
200:	learn: 459.4330264	test: 1163.4573968	best: 1163.4573968 (200)	total: 8.32s	remaining: 16.5s
300:	learn: 343.7453053	test: 1167.5566059	best: 1163.0390039 (203)	total: 14.4s	remaining: 14.3s
400:	learn: 292.6281967	test: 1166.6778950	best: 1163.0390039 (203)	total: 20s	remaining: 9.93s
500:	learn: 256.8142962	test: 1166.1939679	best: 1163.0390039 (203)	total: 25.4s	remaining: 5.01s
599:	learn: 231.6835956	test: 1166.6957448	best: 1163.0390039 (203)	total: 30.5s	remaining: 0us

bestTest = 1163.039004
bestIteration = 203

Shrink model to first 204 iterations.

✅ Model Evaluation:
MAE: 203.15
RMSE: 1352659.72

Sample Predictions:
           State    District       Market    Commodity  Variety   Grade  \
0        Gujarat     Mehsana         Kadi  Castor Seed   Caster   

In [ ]:
import joblib

# ✅ These are the features used for training the model
features = [
    'State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
    'Day', 'Month', 'Year',
    'Min Price', 'Max Price'
]

joblib.dump(features, "/content/drive/MyDrive/MajorProjectModel4_price_prediction/features_list.joblib")

print("✅ features_list.joblib file created successfully!")


✅ features_list.joblib file created successfully!


In [ ]:
from catboost import CatBoostRegressor
import pandas as pd

# Load model
model_path = "/content/drive/MyDrive/MajorProjectModel4_price_prediction/price_model.cbm"
model = CatBoostRegressor()
model.load_model(model_path)

# ✅ Use the exact feature names returned by model.feature_names_
feature_names = model.feature_names_

# ✅ Create a Test Input Row (Fully Correct)
test_input = pd.DataFrame([{
    'State': 'Gujarat',
    'District': 'Amreli',
    'Market': 'Damnagar',
    'Commodity': 'Cabbage',
    'Variety': 'Cabbage',
    'Grade': 'FAQ',
    'Min Price': 2350,
    'Max Price': 3000,
    'price_spread': 3000 - 2350,
    'arrival_day': 27,
    'arrival_month': 7,
    'arrival_year': 2023,
    'arrival_dayofweek': 3
}])[feature_names]

# ✅ Predict
pred = model.predict(test_input)
print("🎯 Predicted Modal Price:", pred[0])


🎯 Predicted Modal Price: 2833.9824676568924
